# Ejercicio 1 — Carga y armonización de datos

In [1]:
import os
import re
from functools import reduce

import openpyxl
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, LongType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab7")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

RAW_DIR = "../working_dir/raw"
PARQUET_DIR = "../working_dir/parquet"
STAGING_DIR = f"{PARQUET_DIR}/staging"

print(spark.version)
print(sorted(os.listdir(RAW_DIR)))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 22:14:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.1
['Base-de-datos-Personas-ENEIC-I-2026.xlsx', 'Base-de-datos-Personas-ENEIC-III-2025.xlsx', 'Base-de-datos-Personas-ENEIC-IV-2025.xlsx', 'Personas-ENEIC-T2-2025.xlsx', 'Personas_ENEIC_T1_2025.xlsx']


## 1.1 Identificación del período de cada archivo

In [2]:
# Metadatos de cada archivo: el período sale del archivo, no de la columna TRIMESTRE
ARCHIVOS = [
    {"archivo": "Personas_ENEIC_T1_2025.xlsx",                "periodo": "2025T1", "anio": 2025, "trimestre": 1},
    {"archivo": "Personas-ENEIC-T2-2025.xlsx",                "periodo": "2025T2", "anio": 2025, "trimestre": 2},
    {"archivo": "Base-de-datos-Personas-ENEIC-III-2025.xlsx", "periodo": "2025T3", "anio": 2025, "trimestre": 3},
    {"archivo": "Base-de-datos-Personas-ENEIC-IV-2025.xlsx",  "periodo": "2025T4", "anio": 2025, "trimestre": 4},
    {"archivo": "Base-de-datos-Personas-ENEIC-I-2026.xlsx",   "periodo": "2026T1", "anio": 2026, "trimestre": 1},
]
pd.DataFrame(ARCHIVOS)

,archivo,periodo,anio,trimestre
0,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
1,Personas-ENEIC-T2-2025.xlsx,2025T2,2025,2
2,Base-de-datos-Personas-ENEIC-III-2025.xlsx,2025T3,2025,3
3,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,2025T4,2025,4
4,Base-de-datos-Personas-ENEIC-I-2026.xlsx,2026T1,2026,1


## 1.2 Selección de columnas y tipos

In [3]:
# Columna original -> nombre analítico
COLUMNAS = {
    "P05D01":      "salario_mensual",
    "P02A03":      "edad",
    "P05C07A":     "antiguedad_anios",
    "P05C07B":     "antiguedad_meses",
    "P05H01A":     "horas_semanales",
    "P03A03A":     "nivel_educativo",
    "P05C16":      "categoria_ocupacional",
    "DOMINIO":     "dominio",
    "OCUPADOS":    "ocupado",
    "NUM_HOGAR":   "NUM_HOGAR",
    "NUM_PERSONA": "NUM_PERSONA",
    "FACTOR":      "FACTOR",
    "ANIO":        "ANIO",
    "TRIMESTRE":   "TRIMESTRE",
}

NUMERICAS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses",
             "horas_semanales", "FACTOR"]
CATEGORICAS = ["nivel_educativo", "categoria_ocupacional", "dominio", "ocupado"]
ENTERAS = {"NUM_HOGAR": LongType(), "NUM_PERSONA": IntegerType(),
           "ANIO": IntegerType(), "TRIMESTRE": IntegerType()}

ESQUEMA = StructType(
    [StructField("archivo_origen", StringType(), False),
     StructField("periodo_archivo", StringType(), False),
     StructField("anio_archivo", IntegerType(), False),
     StructField("trimestre_calendario", IntegerType(), False)]
    + [StructField(c, ENTERAS[c], True) for c in ENTERAS]
    + [StructField(c, StringType(), True) for c in CATEGORICAS]
    + [StructField(c, DoubleType(), True) for c in NUMERICAS]
)
ORDEN_COLUMNAS = [f.name for f in ESQUEMA.fields]

## 1.3 Homologación de códigos

In [4]:
def normalizar_codigo(valor):
    """Convierte un código a texto sin decimales ('5', 5, 5.0 -> '5'). Vacíos -> None."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    if isinstance(valor, (int, float)):
        return str(int(valor)) if float(valor).is_integer() else str(valor)
    texto = str(valor).strip()
    if texto == "":
        return None
    if re.fullmatch(r"-?\d+\.0+", texto):   # '5.0' -> '5'
        texto = texto.split(".")[0]
    return texto  # códigos no numéricos se conservan para validarlos contra el diccionario


def a_numero(serie):
    """Convierte a float; lo que no sea numérico queda como NaN (se contabiliza después)."""
    return pd.to_numeric(serie.map(lambda v: v.strip() if isinstance(v, str) else v),
                         errors="coerce")


def a_valor_spark(v):
    """pandas usa NaN para faltantes; Spark necesita None para que queden como null."""
    if v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA:
        return None
    return v

## 1.4 Carga individual de cada archivo

In [5]:
def contar_columnas_originales(ruta):
    wb = openpyxl.load_workbook(ruta, read_only=True)
    ws = wb[wb.sheetnames[0]]
    n = len(next(ws.iter_rows(min_row=1, max_row=1, values_only=True)))
    wb.close()
    return n


def cargar_archivo(meta):
    ruta = os.path.join(RAW_DIR, meta["archivo"])
    pdf = pd.read_excel(ruta, usecols=list(COLUMNAS), dtype=object)
    pdf = pdf.rename(columns=COLUMNAS)

    for c in NUMERICAS:
        pdf[c] = a_numero(pdf[c]).astype(float).astype(object)
    for c in CATEGORICAS:
        pdf[c] = pdf[c].map(normalizar_codigo)
    for c in ENTERAS:
        pdf[c] = a_numero(pdf[c]).map(lambda v: None if pd.isna(v) else int(v))

    pdf["archivo_origen"] = meta["archivo"]
    pdf["periodo_archivo"] = meta["periodo"]
    pdf["anio_archivo"] = meta["anio"]
    pdf["trimestre_calendario"] = meta["trimestre"]

    filas = [[a_valor_spark(v) for v in fila]
             for fila in pdf[ORDEN_COLUMNAS].itertuples(index=False, name=None)]
    sdf = spark.createDataFrame(filas, schema=ESQUEMA)

    salida = f"{STAGING_DIR}/{meta['periodo']}"
    sdf.write.mode("overwrite").parquet(salida)
    del pdf, filas
    return spark.read.parquet(salida)


resumen_carga = []
bases = {}
for meta in ARCHIVOS:
    ruta = os.path.join(RAW_DIR, meta["archivo"])
    n_cols = contar_columnas_originales(ruta)
    sdf = cargar_archivo(meta)
    bases[meta["periodo"]] = sdf
    resumen_carga.append({"periodo_archivo": meta["periodo"], "archivo_origen": meta["archivo"],
                          "columnas_originales": n_cols, "registros_originales": sdf.count()})
    print(f"{meta['periodo']}: listo")

pd.DataFrame(resumen_carga)

2025T1: listo
2025T2: listo
2025T3: listo
2025T4: listo
2026T1: listo


,periodo_archivo,archivo_origen,columnas_originales,registros_originales
0,2025T1,Personas_ENEIC_T1_2025.xlsx,270,51588
1,2025T2,Personas-ENEIC-T2-2025.xlsx,270,51167
2,2025T3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,270,51583
3,2025T4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,302,49338
4,2026T1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,270,49843


## 1.5 Unión de los archivos de 2025

In [6]:
periodos_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]
df_2025 = reduce(lambda a, b: a.unionByName(b), [bases[p] for p in periodos_2025])
df_2026 = bases["2026T1"]

print("Registros 2025:", df_2025.count())
print("Registros 2026:", df_2026.count())
df_2025.printSchema()

Registros 2025: 203676
Registros 2026: 49843
root
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)
 |-- NUM_HOGAR: long (nullable = true)
 |-- NUM_PERSONA: integer (nullable = true)
 |-- ANIO: integer (nullable = true)
 |-- TRIMESTRE: integer (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)
 |-- ocupado: string (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- FACTOR: double (nullable = true)



### Verificación de la procedencia y de la columna `TRIMESTRE`

In [7]:
(df_2025.unionByName(df_2026)
    .groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE")
    .count()
    .orderBy("periodo_archivo", "TRIMESTRE")
    .show(truncate=False))

+---------------+------------+--------------------+----+---------+-----+
|periodo_archivo|anio_archivo|trimestre_calendario|ANIO|TRIMESTRE|count|
+---------------+------------+--------------------+----+---------+-----+
|2025T1         |2025        |1                   |2025|2        |51588|
|2025T2         |2025        |2                   |2025|2        |175  |
|2025T2         |2025        |2                   |2025|3        |50992|
|2025T3         |2025        |3                   |2025|4        |51583|
|2025T4         |2025        |4                   |2025|5        |49338|
|2026T1         |2026        |1                   |2026|6        |49843|
+---------------+------------+--------------------+----+---------+-----+



### Verificación de la homologación de códigos

Después de normalizar, los códigos categóricos tienen la misma representación en todos los archivos, incluido II de 2025, que venía como texto.

In [8]:
for c in ["categoria_ocupacional", "dominio", "ocupado"]:
    print(f"== {c} ==")
    (df_2025.unionByName(df_2026)
        .groupBy("periodo_archivo").pivot(c).count()
        .orderBy("periodo_archivo")
        .show(truncate=False))

== categoria_ocupacional ==
+---------------+-----+----+----+----+----+----+---+----+---+----+
|periodo_archivo|null |1   |2   |3   |4   |5   |6  |7   |8  |9   |
+---------------+-----+----+----+----+----+----+---+----+---+----+
|2025T1         |29315|1623|6825|4057|914 |5362|497|1625|101|1269|
|2025T2         |28824|1634|7136|3702|1020|5245|649|1713|84 |1160|
|2025T3         |29210|1681|7566|3212|991 |5320|709|1736|123|1035|
|2025T4         |28105|1637|7175|2871|981 |5024|769|1672|110|994 |
|2026T1         |28109|1646|7503|3093|1016|5066|783|1554|112|961 |
+---------------+-----+----+----+----+----+----+---+----+---+----+

== dominio ==
+---------------+-----+-----+-----+
|periodo_archivo|1    |2    |3    |
+---------------+-----+-----+-----+
|2025T1         |16988|19913|14687|
|2025T2         |16928|19821|14418|
|2025T3         |17254|19906|14423|
|2025T4         |16351|18930|14057|
|2026T1         |16905|18952|13986|
+---------------+-----+-----+-----+

== ocupado ==
+--------------

### Esquema y cinco registros de las columnas seleccionadas

In [9]:
df_2025.show(5, truncate=False)

+---------------------------+---------------+------------+--------------------+---------+-----------+----+---------+---------------+---------------------+-------+-------+---------------+----+----------------+----------------+---------------+------+
|archivo_origen             |periodo_archivo|anio_archivo|trimestre_calendario|NUM_HOGAR|NUM_PERSONA|ANIO|TRIMESTRE|nivel_educativo|categoria_ocupacional|dominio|ocupado|salario_mensual|edad|antiguedad_anios|antiguedad_meses|horas_semanales|FACTOR|
+---------------------------+---------------+------------+--------------------+---------+-----------+----+---------+---------------+---------------------+-------+-------+---------------+----+----------------+----------------+---------------+------+
|Personas_ENEIC_T1_2025.xlsx|2025T1         |2025        |1                   |1945     |2          |2025|2        |0              |5                    |1      |1      |NULL           |34.0|2.0             |0.0             |91.0           |47.0  |
|Per

## 1.6 Construcción de la antigüedad

In [10]:
def agregar_antiguedad(df):
    return df.withColumn("antiguedad",
                         F.col("antiguedad_anios") + F.col("antiguedad_meses") / F.lit(12.0))

df_2025 = agregar_antiguedad(df_2025).persist()
df_2026 = agregar_antiguedad(df_2026).persist()

## 1.7 Filtros de población y de calidad

In [11]:
def es_finito(nombre):
    c = F.col(nombre)
    return c.isNotNull() & ~F.isnan(c) & (F.abs(c) != float("inf"))

# (paso, descripción, condición para poder evaluar, condición que debe cumplir)
PASOS_FILTRO = [
    (1, "Edad finita y >= 15",
        es_finito("edad"),
        F.col("edad") >= 15),
    (2, "Ocupado (OCUPADOS = 1)",
        F.col("ocupado").isNotNull(),
        F.col("ocupado") == "1"),
    (3, "Asalariado (P05C16 en 1-4)",
        F.col("categoria_ocupacional").isNotNull(),
        F.col("categoria_ocupacional").isin("1", "2", "3", "4")),
    (4, "Salario finito y > 0",
        es_finito("salario_mensual"),
        F.col("salario_mensual") > 0),
    (5, "Antigüedad en años >= 0",
        es_finito("antiguedad_anios"),
        F.col("antiguedad_anios") >= 0),
    (6, "Meses enteros entre 0 y 11",
        es_finito("antiguedad_meses"),
        (F.col("antiguedad_meses") == F.floor("antiguedad_meses"))
        & F.col("antiguedad_meses").between(0, 11)),
    (7, "Antigüedad <= edad",
        es_finito("antiguedad") & es_finito("edad"),
        F.col("antiguedad") <= F.col("edad")),
    (8, "Horas > 0 y <= 168",
        es_finito("horas_semanales"),
        (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168)),
]


def aplicar_filtros(df):
    """Aplica los pasos en orden y devuelve (df filtrado, bitácora por período y paso)."""
    bitacora = []
    actual = df
    for paso, descripcion, evaluable, cumple in PASOS_FILTRO:
        conteo = (actual
                  .withColumn("_evaluable", evaluable)
                  .withColumn("_cumple", F.col("_evaluable") & F.coalesce(cumple, F.lit(False)))
                  .groupBy("periodo_archivo")
                  .agg(F.count("*").alias("entrada"),
                       F.sum((~F.col("_evaluable")).cast("int")).alias("excluidos_no_evaluable"),
                       F.sum((F.col("_evaluable") & ~F.col("_cumple")).cast("int")).alias("excluidos_no_cumple"),
                       F.sum(F.col("_cumple").cast("int")).alias("restantes"))
                  .toPandas())
        conteo.insert(0, "criterio", descripcion)
        conteo.insert(0, "paso", paso)
        bitacora.append(conteo)
        actual = actual.filter(evaluable & cumple)
    bitacora = pd.concat(bitacora, ignore_index=True)
    bitacora["excluidos_total"] = bitacora["excluidos_no_evaluable"] + bitacora["excluidos_no_cumple"]
    return actual, bitacora.sort_values(["periodo_archivo", "paso"]).reset_index(drop=True)


df_2025_prep, bitacora_2025 = aplicar_filtros(df_2025)
df_2026_prep, bitacora_2026 = aplicar_filtros(df_2026)
df_2025_prep = df_2025_prep.persist()
df_2026_prep = df_2026_prep.persist()
bitacora = pd.concat([bitacora_2025, bitacora_2026], ignore_index=True)

26/09/24 22:17:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### Registros excluidos en cada paso

In [12]:
bitacora["conjunto"] = bitacora["periodo_archivo"].str[:4]
resumen_pasos = (bitacora
    .groupby(["conjunto", "paso", "criterio"], as_index=False)
    [["entrada", "excluidos_no_evaluable", "excluidos_no_cumple", "excluidos_total", "restantes"]]
    .sum())
resumen_pasos["pct_excluido_del_paso"] = (100 * resumen_pasos["excluidos_total"]
                                          / resumen_pasos["entrada"]).round(2)
resumen_pasos

,conjunto,paso,criterio,entrada,excluidos_no_evaluable,excluidos_no_cumple,excluidos_total,restantes,pct_excluido_del_paso
0,2025,1,Edad finita y >= 15,203676,0,62886,62886,140790,30.88
1,2025,2,Ocupado (OCUPADOS = 1),140790,52568,0,52568,88222,37.34
2,2025,3,Asalariado (P05C16 en 1-4),88222,0,35197,35197,53025,39.90
3,2025,4,Salario finito y > 0,53025,0,0,0,53025,0.00
4,2025,5,Antigüedad en años >= 0,53025,0,0,0,53025,0.00
5,2025,6,Meses enteros entre 0 y 11,53025,0,0,0,53025,0.00
6,2025,7,Antigüedad <= edad,53025,0,0,0,53025,0.00
7,2025,8,Horas > 0 y <= 168,53025,0,0,0,53025,0.00
8,2026,1,Edad finita y >= 15,49843,0,14803,14803,35040,29.70
9,2026,2,Ocupado (OCUPADOS = 1),35040,13306,0,13306,21734,37.97


Detalle de registros excluidos por archivo en cada paso:

In [13]:
bitacora.pivot_table(index=["paso", "criterio"], columns="periodo_archivo",
                     values="excluidos_total", aggfunc="sum")

,periodo_archivo,2025T1,2025T2,2025T3,2025T4,2026T1
paso,criterio,,,,,
1,Edad finita y >= 15,16253,15882,15767,14984,14803
2,Ocupado (OCUPADOS = 1),13062,12942,13443,13121,13306
3,Asalariado (P05C16 en 1-4),8854,8851,8923,8569,8476
4,Salario finito y > 0,0,0,0,0,0
5,Antigüedad en años >= 0,0,0,0,0,0
6,Meses enteros entre 0 y 11,0,0,0,0,0
7,Antigüedad <= edad,0,0,0,0,0
8,Horas > 0 y <= 168,0,0,0,0,0


### Registros por archivo antes y después de los filtros

In [14]:
antes = pd.DataFrame(resumen_carga)[["periodo_archivo", "registros_originales"]]
despues = (df_2025_prep.unionByName(df_2026_prep)
           .groupBy("periodo_archivo").count()
           .withColumnRenamed("count", "registros_filtrados")
           .toPandas())
antes_despues = antes.merge(despues, on="periodo_archivo", how="left")
antes_despues["excluidos"] = antes_despues["registros_originales"] - antes_despues["registros_filtrados"]
antes_despues["pct_conservado"] = (100 * antes_despues["registros_filtrados"]
                                   / antes_despues["registros_originales"]).round(2)
antes_despues

,periodo_archivo,registros_originales,registros_filtrados,excluidos,pct_conservado
0,2025T1,51588,13419,38169,26.01
1,2025T2,51167,13492,37675,26.37
2,2025T3,51583,13450,38133,26.07
3,2025T4,49338,12664,36674,25.67
4,2026T1,49843,13258,36585,26.60


## 1.8 Guardado de los conjuntos preparados

In [15]:
RUTA_2025 = f"{PARQUET_DIR}/eneic_2025_preparado"
RUTA_2026 = f"{PARQUET_DIR}/eneic_2026_preparado"

df_2025_prep.write.mode("overwrite").parquet(RUTA_2025)
df_2026_prep.write.mode("overwrite").parquet(RUTA_2026)

# Verificación: se vuelve a leer lo guardado
for ruta in [RUTA_2025, RUTA_2026]:
    print(ruta, "->", spark.read.parquet(ruta).count(), "registros")

../working_dir/parquet/eneic_2025_preparado -> 53025 registros
../working_dir/parquet/eneic_2026_preparado -> 13258 registros
